# 05 - Documentación del Proceso Analítico

Este notebook documenta el flujo completo del proyecto de análisis exploratorio de dengue en Colombia (2010-2024). No introduce análisis nuevos — consolida las decisiones, hallazgos y limitaciones de los notebooks 00-04 como referencia técnica del repositorio.

**Propósito:** Servir como complemento técnico al artículo académico, documentando el pipeline reproducible desde los datos crudos hasta el dataset procesado.

## 1. Objetivo y Alcance del Proyecto

**Objetivo general:** Comprender la estructura, calidad y patrones de un panel epidemiológico municipio-semana de dengue en Colombia (2010-2024) mediante análisis exploratorio descriptivo.

**Alcance:**
- Evaluación de calidad, completitud e integridad del dataset.
- Caracterización univariada, bivariada y multivariada de variables epidemiológicas, climáticas, ambientales y de movilidad.
- Detección, clasificación y documentación de valores extremos.
- Preprocesamiento: imputación, transformaciones y creación de variables derivadas.

**Fuera de alcance:** No se realizan modelos predictivos, forecasting, inferencia causal ni machine learning supervisado. El enfoque es exclusivamente descriptivo y exploratorio.

## 2. Pipeline Analítico

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd

# Estructura del pipeline
pipeline = pd.DataFrame([
    ["00", "Data Understanding", "raw_dataset.csv (813,280 × 22)", "Validación de panel, duplicados, faltantes, integridad lógica, diccionario de variables", "tables/data_understanding/"],
    ["01", "EDA Univariado", "raw_dataset.csv", "Distribuciones, ceros, asimetría, colas; por grupo temático (epidemiológicas, climáticas, ambientales, demográficas)", "figures/eda_univariate/, tables/eda_univariate/"],
    ["02", "EDA Multivariado", "final_dataset.csv (294,032 × 25)", "Correlaciones Pearson/Spearman, OLS exploratorio, VIF, comparaciones por grupo (departamento, ONI, temperatura)", "figures/eda_multivariate/, tables/eda_multivariate/"],
    ["03", "Outliers", "final_dataset.csv", "IQR, Z-score MAD, Isolation Forest, LOF; clasificación y decisión de tratamiento", "figures/outliers/, tables/outliers/"],
    ["04", "Preprocessing", "final_dataset.csv", "Evaluación de imputación, tratamiento de ceros, transformaciones log, lags, rolling means, features estacionales", "data/processed/dataset_processed.csv"],
], columns=["Notebook", "Etapa", "Input", "Qué hace", "Outputs principales"])

display(Markdown("### Flujo de notebooks"))
display(pipeline)

### Diagrama de flujo

```
data/raw/raw_dataset.csv (1,040 municipios × 782 semanas)
        │
        ▼
    [00] Data Understanding ──→ Validación estructural, diccionario, alertas
        │
        ▼
    [01] EDA Univariado ──→ Distribuciones por variable, problemas identificados
        │
        ▼
data/raw/final_dataset.csv (376 municipios × 782 semanas, imputado)
        │
        ├──→ [02] EDA Multivariado ──→ Correlaciones, OLS, VIF, grupos
        │
        ├──→ [03] Outliers ──→ Detección multi-método, decisiones de tratamiento
        │
        ▼
    [04] Preprocessing ──→ Transformaciones, lags, features
        │
        ▼
data/processed/dataset_processed.csv (376 × 770 semanas, 43 columnas)
```

**Datasets utilizados:**
- `raw_dataset.csv`: Panel completo (1,040 municipios), con faltantes reales.
- `final_dataset.csv`: Subconjunto filtrado (376 municipios con cobertura completa), con imputación previa de `temp_mean`.
- `dataset_processed.csv`: Dataset final con transformaciones y features temporales, listo para análisis.

## 3. Descripción del Dataset

In [ ]:
dataset_description = pd.DataFrame([
    ["Fuente", "Panel epidemiológico de dengue, clima, ambiente y movilidad para Colombia (Kaggle/fuente externa)"],
    ["Unidad de observación", "Municipio × semana epidemiológica"],
    ["Periodo", "2010-2024 (782 semanas epidemiológicas)"],
    ["Cobertura geográfica (raw)", "1,040 municipios (códigos DIVIPOLA)"],
    ["Cobertura geográfica (final)", "376 municipios con cobertura temporal completa"],
    ["Variable principal", "casos_totales: conteo semanal de casos de dengue por municipio"],
    ["Variables epidemiológicas", "Estratificación por edad (4 grupos), sexo (2), estrato socioeconómico (3)"],
    ["Variables climáticas", "temp_mean (temperatura media °C), prec_total (precipitación mm)"],
    ["Variables ambientales", "ndvi_mean (vegetación), oni_anom y oni_total (índice oceánico El Niño)"],
    ["Variables de movilidad", "Flujo_in (movilidad entrante al municipio)"],
    ["Variables demográficas", "poblacion (población municipal anual)"],
], columns=["Característica", "Descripción"])

display(Markdown("### Ficha técnica del dataset"))
display(dataset_description)

## 4. Resumen de Decisiones Metodológicas

In [ ]:
decisions = pd.DataFrame([
    ["Validación de panel", "00", "Verificar estructura municipio-semana sin duplicados", "Panel 100% balanceado, 0 duplicados, integridad lógica aprobada (6/6 validaciones)."],
    ["No eliminar faltantes climáticos", "00-01", "Documentar sin imputar en raw; el final_dataset ya viene imputado", "temp_mean con 78% faltantes en raw; en final_dataset, 64% imputado con media global (baja calidad)."],
    ["Análisis en escala log1p", "01-02", "Reducir asimetría extrema (skewness 21-33) para visualizaciones y correlaciones", "Transformación monotónica reversible; no altera rankings ni estructura de relaciones."],
    ["Spearman sobre Pearson", "02", "Las relaciones son monótonas pero no lineales (Spearman > Pearson para Flujo_in y poblacion)", "Flujo_in: Pearson 0.35 → Spearman 0.51. Relaciones de umbral y saturación."],
    ["No eliminar outliers epidemiológicos", "03", "Los valores extremos de casos son brotes reales, no errores", "IQR marca como outlier toda incidencia >5 casos. Los extremos son el fenómeno a estudiar."],
    ["Flag outlier_consensus", "03", "Documentar con flag para sensibilidad, no para eliminación", "3.7% del dataset es outlier por ≥2 métodos. Son principalmente ciudades endémicas grandes."],
    ["Flag temp_reliable", "04", "226 municipios (60%) tienen temp_mean imputada con valor constante", "Se marca como no confiable para análisis climático. Solo 150 municipios tienen datos reales."],
    ["Imputar Flujo_in = 0", "04", "752 ceros exactamente en 2 fechas (artefacto de calibración)", "Imputado con mínimo no-cero del municipio. Ceros de casos y precipitación se mantienen."],
    ["Crear lags y rolling means", "04", "Capturar inercia epidémica (autocorrelación dominante)", "lag1 ρ=0.98; las variables de historia reciente superan cualquier predictor externo."],
    ["Warm-up period de 12 semanas", "04", "Eliminar NaN generados por operaciones de lag", "Pérdida menor (1.5%) uniforme entre municipios, sin sesgo."],
], columns=["Decisión", "Notebook", "Justificación", "Resultado/Impacto"])

display(Markdown("### Decisiones clave del proceso"))
display(decisions)

## 5. Hallazgos Principales Consolidados

In [ ]:
hallazgos = pd.DataFrame([
    ["Estructura epidemiológica", "El dengue en Colombia es zero-inflated: 57-78% de municipios-semana sin casos. La mediana es 0, pero el P99 es 32-75 casos.", "01, 04"],
    ["Movilidad como driver", "Flujo_in es el predictor bivariado más fuerte (ρ=0.51 Spearman, coef OLS=0.80). La conectividad urbana domina la transmisión.", "02"],
    ["ENSO como modulador temporal", "El Niño triplica la tasa de incidencia respecto a La Niña. oni_anom es ortogonal al resto (VIF=1.08): aporta información temporal no redundante.", "02"],
    ["Multicolinealidad geográfica", "VIF severo: log_poblacion=76, temp_mean=53, ndvi_mean=28. Refleja la estructura altitud→temperatura→urbanización→vegetación de Colombia.", "02"],
    ["Autocorrelación temporal extrema", "La historia reciente predice el presente mejor que cualquier variable externa. DW=0.526 en OLS; lag1 ρ=0.98.", "02, 04"],
    ["Temperatura: condición necesaria, no suficiente", "Asociación positiva débil (ρ=0.10) porque opera como umbral (<20°C sin transmisión) y está confundida con urbanización.", "02"],
    ["Precipitación: no discrimina en agregado", "ρ<0.01 en cualquier lag. El mecanismo real es frecuencia de lluvias cortas, no acumulado semanal.", "02, 04"],
    ["Imputación de temp_mean: baja calidad", "60% del panel recibió un valor constante (~22.4°C). Variabilidad comprimida 7x respecto a datos reales.", "04"],
    ["Outliers = ciudades endémicas", "Isolation Forest captura nodos urbanos permanentes (Bogotá, Medellín, Cali). No son errores: son la columna vertebral epidemiológica.", "03"],
    ["OLS explica 46.5% con 6 variables", "Base sólida. El 53.5% restante es dinámica epidemiológica no capturada (inmunidad, serotipos, control vectorial).", "02"],
], columns=["Hallazgo", "Evidencia", "Notebooks"])

display(Markdown("### Consolidación de hallazgos"))
display(hallazgos)

## 6. Limitaciones y Consideraciones Éticas

In [ ]:
limitaciones = pd.DataFrame([
    ["Imputación previa opaca", "La imputación de temp_mean en final_dataset.csv fue externa y destruyó la variabilidad climática real en 60% de municipios.", "Se creó flag temp_reliable; resultados climáticos aplican solo a 150 municipios con datos reales."],
    ["Subnotificación no medible", "Los ceros pueden representar ausencia real de dengue o falla del sistema de vigilancia.", "Se asumen como ceros reales dado que el panel tiene cobertura completa. No es posible distinguir sin datos del sistema SIVIGILA."],
    ["Dataset observacional", "No hay asignación experimental. Las correlaciones encontradas no implican causalidad.", "Todas las asociaciones (Flujo_in→casos, ONI→casos) se reportan como tales, sin lenguaje causal."],
    ["Exclusión de 664 municipios", "El final_dataset filtra a 376 municipios, excluyendo los que carecen de cobertura temporal completa.", "Los municipios excluidos son predominantemente rurales y pequeños. Los hallazgos no son generalizables al 100% del territorio."],
    ["Ausencia de variables clave", "No se dispone de: serotipos circulantes, índices entomológicos, cobertura de intervenciones de control vectorial, inmunidad poblacional.", "El R²=0.465 del OLS refleja esta limitación. Los modelos predictivos futuros necesitarían estas variables."],
    ["Ética en datos epidemiológicos", "El dataset es agregado (municipio-semana) sin identificación individual. No hay riesgo de re-identificación.", "Se citan fuentes de datos; no se modifican ni se imputan valores que puedan alterar la magnitud real de la enfermedad."],
], columns=["Limitación", "Descripción", "Mitigación"])

display(Markdown("### Limitaciones del análisis"))
display(limitaciones)

## 7. Líneas Futuras y Recomendaciones

El trabajo descriptivo realizado en este repositorio sienta las bases para extensiones analíticas que quedan fuera del alcance actual:

1. **Modelado predictivo:** La estructura autorregresiva (lag1 ρ=0.98) y los predictores identificados (Flujo_in, ONI, temperatura) sugieren que modelos basados en árboles (XGBoost) o modelos de series de tiempo con covariables externas podrían predecir brotes con horizonte de 1-4 semanas.

2. **Ingeniería de features climáticos:** La precipitación no aporta en su forma cruda. Variables derivadas como días sin lluvia, frecuencia de eventos, o índices de sequía podrían capturar el mecanismo real de formación de criaderos.

3. **Estructura espacial:** La multicolinealidad geográfica (VIF>50) indica que los efectos de temperatura, urbanización y vegetación están confundidos espacialmente. Modelos jerárquicos o con efectos espaciales podrían descomponer estas contribuciones.

4. **Validación externa de imputación:** Obtener datos satelitales (ERA5, CHIRPS) para los 226 municipios sin temperatura real permitiría reemplazar la imputación con media global por valores locales creíbles.

5. **Análisis temporal formal:** Descomposición STL, canal endémico y detección de brotes permitirían cuantificar la estructura estacional y los periodos anómalos identificados cualitativamente en este trabajo.

## 8. Herramientas y Dependencias

In [ ]:
herramientas = pd.DataFrame([
    ["pandas", "Manipulación de datos, panel, agrupaciones", "Todos"],
    ["numpy", "Operaciones numéricas, transformaciones", "Todos"],
    ["matplotlib", "Visualizaciones base", "Todos"],
    ["seaborn", "Visualizaciones estadísticas (heatmaps, violin, boxplots)", "01, 02, 03, 04"],
    ["scipy.stats", "Asimetría, curtosis, correlaciones, tests", "00, 01, 02"],
    ["scikit-learn", "Isolation Forest, LOF, StandardScaler", "03"],
    ["statsmodels", "OLS, VIF, diagnósticos de regresión", "02"],
    ["jupyter/notebook", "Entorno de ejecución interactiva", "Todos"],
], columns=["Librería", "Uso", "Notebooks"])

display(Markdown("### Stack tecnológico"))
display(herramientas)

print("Python 3.13 | Entorno virtual (.venv) | Repositorio Git")

## 9. Mapa de Outputs del Repositorio

In [ ]:
import os

output_dirs = [
    ("tables/data_understanding/", "Notebook 00"),
    ("tables/eda_univariate/", "Notebook 01"),
    ("tables/eda_multivariate/", "Notebook 02"),
    ("tables/outliers/", "Notebook 03"),
    ("tables/preprocessing/", "Notebook 04"),
]

print("=== Tablas generadas ===")
for dir_path, source in output_dirs:
    full_path = Path("..") / dir_path
    if full_path.exists():
        files = sorted(full_path.glob("*.csv"))
        print(f"\n{dir_path} ({source}):")
        for f in files:
            print(f"  {f.name}")
    else:
        print(f"\n{dir_path} ({source}): [directorio no encontrado]")

print("\n=== Datos procesados ===")
processed = Path("../data/processed")
if processed.exists():
    for f in sorted(processed.glob("*")):
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"  {f.name} ({size_mb:.1f} MB)")